# Weight Loading
Description
Having processed our text we will be start by implmenting the training loop and code out a basic model evaluation to pretrain an LLM

Prereqs before the lab: Build a Large Language Model (From Scratch) ch4 (video or textformat)

Citation Raschka, S. (2024). Build a large language model (from scratch). Manning Publications.

Lab Deliverables Read though: https://github.com/rasbt/LLM-workshop-2024/blob/main/05_weightloading/05_part-1.ipynb

Set up Be sure to upload the supplementry and gpt_download files  provided in the assigment page on canvas.

Directions:

Step 1.
- Import `download_and_load_gpt2` from the provided file `gpt_download`.    
- Call `download_and_load_gpt2` with the arguments `model_size="123M"` and `models_dir="gpt2"`. Store the returned values in the variables `settings` and `params`.  
- Print both `settings` and `params`. When printing `params`, use `.keys()` to display the parameter names.  
- Print `params["wte"].shape` to view the shape of the embedding weight matrix.  
<details>
<summary><strong>Solution</strong></summary>

```
from gpt_download import download_and_load_gpt2
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

```

</details>

In [3]:
from gpt_download import download_and_load_gpt2
model_tag = "124M"
models_dir = "gpt2"
settings, params = download_and_load_gpt2(model_size=model_tag, models_dir=models_dir)

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 51.5kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 3.24MiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 42.2kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [00:43<00:00, 11.3MiB/s] 
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 5.15MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:00<00:00, 2.10MiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:00<00:00, 2.02MiB/s]


In [4]:
print("Settings:", settings)
print("Top-level parameter groups:", list(params.keys()))


Settings: {'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
Top-level parameter groups: ['blocks', 'b', 'g', 'wpe', 'wte']


In [ ]:

print("Embedding matrix shape:", params["wte"].shape)


Embedding matrix shape: (50257, 768)


In [7]:
print(params["wte"])

[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]


## Step 2: Transfer GPT-2 weights into a `GPTModel` instance

- Define the base parameters:
  - `"vocab_size": 50257` (Vocabulary size)
  - `"context_length": 256` (Shortened context length, original is 1024)
  - `"emb_dim": 768` (Embedding dimension)
  - `"n_heads": 12` (Number of attention heads)
  - `"n_layers": 12` (Number of layers)
  - `"drop_rate": 0.1` (Dropout rate)
  - `"qkv_bias": False` (Query-key-value bias)

- Define the model size configurations in a dictionary called `model_configs` to keep things clean and compact:
  - `"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12}`
  - `"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16}`
  - `"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20}`
  - `"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}`

- Give the model a name (pick one of the keys from `model_configs`).
- Make a copy of the selected config into a variable called `NEW_CONFIG`.
- Add the model name to `NEW_CONFIG`.
- Also add these keys to `NEW_CONFIG`:
  - `"context_length": 1024`
  - `"qkv_bias": True`

- Import `GPTModel`.
- Initialize a `GPTModel` instance using `NEW_CONFIG`.
- Call `.eval()` on the new model.

<details>
<summary><strong>Solution</strong></summary>

  ```
  GPT_CONFIG_124M = {
      "vocab_size": 50257,   # Vocabulary size
      "context_length": 256, # Shortened context length (orig: 1024)
      "emb_dim": 768,        # Embedding dimension
      "n_heads": 12,         # Number of attention heads
      "n_layers": 12,        # Number of layers
      "drop_rate": 0.1,      # Dropout rate
      "qkv_bias": False      # Query-key-value bias
  }


  # Define model configurations in a dictionary for compactness
  model_configs = {
      "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
      "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
      "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
      "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
  }

  # Copy the base configuration and update with specific model settings
  model_name = "gpt2-small (124M)"  # Example model name
  NEW_CONFIG = GPT_CONFIG_124M.copy()
  NEW_CONFIG.update(model_configs[model_name])
  NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

  from supplementary import GPTModel

  gpt = GPTModel(NEW_CONFIG)
  gpt.eval();
  ```
</details>

In [8]:
base_config = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


In [9]:
from supplementary import GPTModel

selected_model = "gpt2-small (124M)"
GPT_CONFIG_124M = {**base_config, **model_configs[selected_model]}
GPT_CONFIG_124M["context_length"] = 1024
GPT_CONFIG_124M["qkv_bias"] = True

gpt = GPTModel(GPT_CONFIG_124M)
gpt.eval()



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\rezah\anaconda3\envs\school\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\rezah\anaconda3\envs\school\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\rezah\anaconda3\envs\school\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

## Step 3: Assign the OpenAI weights to the corresponding tensors in the GPT model

- Create a function called `assign` that takes two parameters: `left` and `right`.
  - Check if `left.shape` is not equal to `right.shape`.
    - If the shapes do not match, raise this error:
      ```python
      ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
      ```
    - This ensures that only tensors with compatible dimensions are assigned.

  - If the shapes match, return the `right` tensor converted into a parameter suitable for the model.
    - This step transfers the pretrained OpenAI weights into the corresponding location in the GPT model.
    - It ensures the model uses the pretrained values instead of its randomly initialized weights.

- This next section is more complex. Instead of writing code, your task is to read the provided code and explain what it is doing.
  - You will see `# Comments` markers in the code. Replace each marker with your own explanation.
  - Each comment should briefly explain what the code is doing at that point. Aim for 1 to 3 sentences per comment.
  - Focus on understanding how the pretrained OpenAI weights are being assigned to the corresponding parts of the GPT model.
  - If you are unsure, you may use an AI tool to help you understand the code before writing your explanation. However, make sure your final comments reflect your own understanding.

<details>
<summary><strong>Solution</strong></summary>
#This is only for the first part, there's no soluiton for the comments.

```
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))
```

</details>


In [10]:
import torch

def assign(left, right):
    incoming = torch.as_tensor(right, dtype=left.dtype, device=left.device)
    if tuple(left.shape) != tuple(incoming.shape):
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {incoming.shape}")
    return torch.nn.Parameter(incoming.clone().detach())


In [12]:

import torch
import numpy as np

def load_weights_into_gpt(gpt, params):
  #Comments Here:
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

      #Comments Here:
    for b in range(len(params["blocks"])):

        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])
  #Comments Here:
    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

#Comments Here:
load_weights_into_gpt(gpt, params)
#Comments Here:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

## Step 4: Test that the model is loaded correctly

- Import `tiktoken`.
- Import the following functions from the `supplementary` file:
  - `generate_text_simple`
  - `text_to_token_ids`
  - `token_ids_to_text`

- Initialize the tokenizer using `.get_encoding()` and pass `"gpt2"` as the argument.

- Set the manual seed to `123` to ensure reproducibility. This makes sure the model generates the same output each time it runs.

- Use `generate_text_simple` to generate text with the following parameters:
  - `model=gpt`
  - `idx=text_to_token_ids("Every effort moves you", tokenizer).to(device)`
  - `max_new_tokens=10`
  - `context_size=GPT_CONFIG_124M["context_length"]`

- Convert the generated token IDs back into human-readable text using `token_ids_to_text`.

- Verify that the output is coherent. This confirms that the pretrained weights were loaded correctly and the model is functioning properly.

<details>
<summary><strong>Solution</strong></summary>

```
import tiktoken
from supplementary import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(123)

token_ids = generate_text_simple(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))
```

</details>

In [13]:
import torch
import tiktoken
from supplementary import generate_text_simple, text_to_token_ids, token_ids_to_text

encoder = tiktoken.get_encoding("gpt2")
torch.manual_seed(123)

seed_prompt = "Every effort moves you"
input_tokens = text_to_token_ids(seed_prompt, encoder).to(device)

generated_tokens = generate_text_simple(
    model=gpt,
    idx=input_tokens,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

generated_text = token_ids_to_text(generated_tokens, encoder)
print(f"Model check output:\n{generated_text}")

Model check output:
Every effort moves you forward.

The first step is to understand
